# Task-Based Execution with the Python API

When using `openfe quickrun`, you orchestrate the campaign. You choose which `Transformation` to run and when to run it.

With task-based execution, **openfe** handles that orchestration for you. You provide an entire `AlchemicalNetwork`, **openfe** breaks it into dependent tasks, and one or more `Worker`s claim tasks as soon as they are ready to run.

A **task** is a single `ProtocolUnit`, for example the setup, simulation, or
analysis step of one repeat of one `Transformation`.

Three objects make up a task-based campaign:

- the **`Warehouse`** stores the campaign data needed for execution, including
  the `AlchemicalNetwork`, tasks, and results;
- the **`TaskStatusDB`** tracks task status and dependencies;
- a **`Worker`** claims an available task, executes it, and stores its result.

Separating campaign data, orchestration state, and execution allows the same
campaign to be executed by multiple workers and across multiple processes.

This tutorial walks through a task-based campaign using the Python API.
For the equivalent command-line workflow, see the
[Task-based Execution section of the OpenFE user guide](https://docs.openfree.energy/en/latest/guide/execution/task_based_execution.html).

<img src="execution.png" alt="Drawing" style="width: 800px;"/>

> **Note:** To run this tutorial, clone the OpenFE ExampleNotebooks repository and run the notebook from its repository directory. The example input file used below is included in the repository.

In [1]:
from pathlib import Path

import openfe

# use the below lines to clear the files created by this notebook and start fresh:
# ! rm tasks_mcl1.db
# ! rm -rf warehouse_mcl1/ mcl1_result_jsons/

## Load an AlchemicalNetwork

The input to a task-based campaign is an `AlchemicalNetwork`. If you don't
have one yet, see the
[Create an `AlchemicalNetwork` cookbook](https://docs.openfree.energy/en/stable/cookbook/create_alchemical_network.html),
or add `--networks-only` to an `openfe plan-rbfe-network` call.

Here we use a small MCL-1 network with pre-charged ligands and two edges,
`ligand_1 → ligand_2` and `ligand_2 → ligand_3`. Each edge has a complex and
a solvent leg, giving four `Transformation`s.

For this example, each `Transformation` has one repeat. Each repeat of the
hybrid-topology protocol consists of setup, simulation, and analysis units,
so the campaign will contain 12 tasks in total.

In [2]:
alchemical_network = openfe.AlchemicalNetwork.from_json("alchemicalNetwork_mcl1_small/alchemicalNetwork_mcl1_small.json")

## Set up the campaign

`setup_task_campaign()` prepares the `AlchemicalNetwork` for task-based execution by creating two resources on disk that workers can reuse later:

- a **Warehouse**, which stores the network, tasks, DAG information, shared
  files, and results;
- a **TaskStatusDB**, which stores task status and dependency information.

This is equivalent to

`openfe setup-task-campaign --alchemical-network alchemicalNetwork_mcl1_small.json`

on the command line.

> **Tip:** We recommend prefixing Warehouse directories with `warehouse_` and task database
> files with `tasks_`. This follows the CLI defaults and makes the files belonging
> to a campaign easier to recognize.

In [3]:
from openfe.orchestration import setup_task_campaign

campaign_name = "mcl1"
warehouse_dir = f"warehouse_{campaign_name}"
db_path = f"tasks_{campaign_name}.db"

task_db, wh = setup_task_campaign(
    alchemical_network=alchemical_network,
    warehouse_dir=warehouse_dir,
    db_path=db_path,
)

In [4]:
print(f"Warehouse:     {wh}")
print(f"Task database: {task_db}")

Warehouse:     FileSystemWarehouse({'setup': <gufe.storage.externalresource.filestorage.FileStorage object at 0x31ffbb620>, 'results': <gufe.storage.externalresource.filestorage.FileStorage object at 0x3208416d0>, 'shared': <gufe.storage.externalresource.filestorage.FileStorage object at 0x320841810>, 'tasks': <gufe.storage.externalresource.filestorage.FileStorage object at 0x16844f820>, 'protocol_dags': <gufe.storage.externalresource.filestorage.FileStorage object at 0x31fc83bb0>})
Task database: TaskStatusDB(engine=Engine(sqlite:///tasks_mcl1.db))


### The Warehouse

The `Warehouse` stores the data needed to execute the campaign and the results produced during execution. Its directory contains separate stores for:

- `setup`: the `AlchemicalNetwork` and the objects needed to define the campaign;
- `tasks`: the `ProtocolUnit`s that workers execute;
- `protocol_dags`: information describing the `ProtocolDAG`s used to organize and reconstruct results;
- `results`: results produced by individual task executions;
- `shared`: files that need to persist between dependent `ProtocolUnit`s.

These files and folders should **not** be modified by the user.

In [5]:
sorted(path.name for path in Path(warehouse_dir).iterdir())

['protocol_dags', 'results', 'setup', 'shared', 'tasks']

In [6]:
# the AlchemicalNetwork is now stored in the Warehouse
wh.exists(alchemical_network.key)

True

### The TaskStatusDB

The `TaskStatusDB` stores the orchestration state of the campaign.
`get_task_df` returns the task table as a `pandas.DataFrame`, which is the same
information shown by the `openfe status` command.

For this network, each of the four `Transformation`s has one repeat, and each repeat contains three units, giving 12 tasks in total.

Initially, only the four setup units are `AVAILABLE`. The simulation and
analysis units are `BLOCKED` because their dependencies have not completed yet.

As execution proceeds you will also see tasks become `IN_PROGRESS` and
`COMPLETED`. Failed execution can additionally produce error/retry-related
states such as `ERROR` and `TOO_MANY_RETRIES`. The `tries` and `max_tries`
columns track retry accounting.

In [7]:
from openfe.orchestration import get_task_df, get_dependency_df

task_df = get_task_df(task_db)
task_df 

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-b11e2040225240bf97425b...,AVAILABLE,NaT,0,1
1,HybridTopologySetupUnit-466a9387de15407083aa5c...,AVAILABLE,NaT,0,1
2,HybridTopologySetupUnit-bc42239bbff74669b323c3...,AVAILABLE,NaT,0,1
3,HybridTopologySetupUnit-cdb0d6a759aa42beb064bb...,AVAILABLE,NaT,0,1
4,HybridTopologyMultiStateSimulationUnit-6df05e9...,BLOCKED,NaT,0,1
5,HybridTopologyMultiStateSimulationUnit-d12fa30...,BLOCKED,NaT,0,1
6,HybridTopologyMultiStateSimulationUnit-f8664b8...,BLOCKED,NaT,0,1
7,HybridTopologyMultiStateSimulationUnit-7acd26b...,BLOCKED,NaT,0,1
8,HybridTopologyMultiStateAnalysisUnit-9c2cbafed...,BLOCKED,NaT,0,1
9,HybridTopologyMultiStateAnalysisUnit-1757c7940...,BLOCKED,NaT,0,1


`get_dependency_df` shows how tasks depend on one another. Read each row as:
the task in `from` must complete before the task in `to` can run.

Each simulation unit depends on its corresponding setup unit. Each analysis
unit depends on the setup and simulation results required for that
`Transformation`.

In [8]:
dep_df = get_dependency_df(task_db)
dep_df

,from,to,blocking
0,HybridTopologySetupUnit-b11e2040225240bf97425b...,HybridTopologyMultiStateSimulationUnit-6df05e9...,True
1,HybridTopologySetupUnit-466a9387de15407083aa5c...,HybridTopologyMultiStateSimulationUnit-d12fa30...,True
2,HybridTopologySetupUnit-bc42239bbff74669b323c3...,HybridTopologyMultiStateSimulationUnit-f8664b8...,True
3,HybridTopologySetupUnit-cdb0d6a759aa42beb064bb...,HybridTopologyMultiStateSimulationUnit-7acd26b...,True
4,HybridTopologySetupUnit-b11e2040225240bf97425b...,HybridTopologyMultiStateAnalysisUnit-9c2cbafed...,True
5,HybridTopologyMultiStateSimulationUnit-6df05e9...,HybridTopologyMultiStateAnalysisUnit-9c2cbafed...,True
6,HybridTopologySetupUnit-466a9387de15407083aa5c...,HybridTopologyMultiStateAnalysisUnit-1757c7940...,True
7,HybridTopologyMultiStateSimulationUnit-d12fa30...,HybridTopologyMultiStateAnalysisUnit-1757c7940...,True
8,HybridTopologySetupUnit-bc42239bbff74669b323c3...,HybridTopologyMultiStateAnalysisUnit-36d32ecda...,True
9,HybridTopologyMultiStateSimulationUnit-f8664b8...,HybridTopologyMultiStateAnalysisUnit-36d32ecda...,True


**Tip:** use the pandas DataFrame ``.value_counts()`` method to get a summary of the status.

In [9]:
task_df["status"].value_counts()

status
BLOCKED      8
AVAILABLE    4
Name: count, dtype: int64

## Execute tasks with a `Worker`

A `Worker` connects to the `Warehouse` and the `TaskStatusDB` to execute tasks from the campaign.

Each call to `Worker.execute_unit`:

1. claims the next `AVAILABLE` task from the `TaskStatusDB` and marks it `IN_PROGRESS`
2. loads the corresponding `ProtocolUnit` and any required upstream results from the
   `Warehouse`
3. executes the `ProtocolUnit`, using `scratch` for temporary working files
4. stores the results in the `Warehouse` and updates the task status in the TaskStatusDB

When a task completes successfully, any downstream tasks whose dependencies are now satisfied become `AVAILABLE`.

`execute_unit` returns information about the task it executed, or `None` if no task is available to claim.

In [10]:
from openfe.orchestration import Worker

worker = Worker(wh, db_path)

### Execute one task

Here, we will execute **one** task to see how task states change as dependencies are satisfied.

Notice that we do not choose which task the worker execute.
`Worker.execute_unit` claims an `AVAILABLE` task from the `TaskStatusDB`. At this point, all available tasks are setup units.

In [11]:
exec_result = worker.execute_unit(scratch="scratch")
exec_result

INFO:openfe.utils.system_probe.log:SYSTEM CONFIG DETAILS:
INFO:openfe.utils.system_probe.log.hostname:hostname: 'Alyssas-Macbook-Pro.local'
INFO:openfe.utils.system_probe.log.gpu:CUDA-based GPU not found
INFO:openfe.utils.system_probe.log:Memory used: 20.8G (63.9%)
INFO:openfe.utils.system_probe.log:scratch/task_workdirs/HybridTopologySetupUnit-b11e2040225240bf97425ba0e62b54ac: 73% full (251.6G free)
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Starting system setup unit
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Parameterizing systems
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][C]1=[C]([C](=[O])[O-])[S][c]2[c]([H])[c]([H])[c]([H])[c]([H])[c]21 using openff-2.2.1
/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:171: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_

('HybridTopologySetupUnit-b11e2040225240bf97425ba0e62b54ac',
 ProtocolUnitResult(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))

After the setup task completes, we inspect the task database again.

The completed setup unit is now `COMPLETED`. Its corresponding simulation unit
has changed from `BLOCKED` to `AVAILABLE`, while unrelated tasks retain their
previous state.

In [12]:
task_df = get_task_df(task_db)
task_df

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-b11e2040225240bf97425b...,COMPLETED,2026-10-06 17:34:13.240953,1,1
1,HybridTopologySetupUnit-466a9387de15407083aa5c...,AVAILABLE,NaT,0,1
2,HybridTopologySetupUnit-bc42239bbff74669b323c3...,AVAILABLE,NaT,0,1
3,HybridTopologySetupUnit-cdb0d6a759aa42beb064bb...,AVAILABLE,NaT,0,1
4,HybridTopologyMultiStateSimulationUnit-6df05e9...,AVAILABLE,2026-10-06 17:34:13.241104,0,1
5,HybridTopologyMultiStateSimulationUnit-d12fa30...,BLOCKED,NaT,0,1
6,HybridTopologyMultiStateSimulationUnit-f8664b8...,BLOCKED,NaT,0,1
7,HybridTopologyMultiStateSimulationUnit-7acd26b...,BLOCKED,NaT,0,1
8,HybridTopologyMultiStateAnalysisUnit-9c2cbafed...,BLOCKED,NaT,0,1
9,HybridTopologyMultiStateAnalysisUnit-1757c7940...,BLOCKED,NaT,0,1


**Tip:** You can use the ``update_max_tries()`` method to change the `max_tries` for all valid rows.

(Rows that are marked ``COMPLETED`` or whose ``tries`` are greater than the provided value will not be updated)

In [13]:
openfe.update_max_tries(task_db=task_db, max_tries=2)

get_task_df(task_db)

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-b11e2040225240bf97425b...,COMPLETED,2026-10-06 17:34:13.240953,1,1
1,HybridTopologySetupUnit-466a9387de15407083aa5c...,AVAILABLE,2026-10-06 17:34:13.276992,0,2
2,HybridTopologySetupUnit-bc42239bbff74669b323c3...,AVAILABLE,2026-10-06 17:34:13.276992,0,2
3,HybridTopologySetupUnit-cdb0d6a759aa42beb064bb...,AVAILABLE,2026-10-06 17:34:13.276992,0,2
4,HybridTopologyMultiStateSimulationUnit-6df05e9...,AVAILABLE,2026-10-06 17:34:13.276992,0,2
5,HybridTopologyMultiStateSimulationUnit-d12fa30...,BLOCKED,2026-10-06 17:34:13.276992,0,2
6,HybridTopologyMultiStateSimulationUnit-f8664b8...,BLOCKED,2026-10-06 17:34:13.276992,0,2
7,HybridTopologyMultiStateSimulationUnit-7acd26b...,BLOCKED,2026-10-06 17:34:13.276992,0,2
8,HybridTopologyMultiStateAnalysisUnit-9c2cbafed...,BLOCKED,2026-10-06 17:34:13.276992,0,2
9,HybridTopologyMultiStateAnalysisUnit-1757c7940...,BLOCKED,2026-10-06 17:34:13.276992,0,2


### Running the rest of a campaign

For a campaign, a worker can repeatedly execute tasks until no work is available:

```python
while True:
    result = worker.execute_unit(scratch="scratch")
    if result is None:
        break
```

In production, several independent workers can run in separate jobs on an HPC system, all operating on the same campaign, to run tasks in parallel.
See the [task-based execution User Guide](https://docs.openfree.energy/en/latest/guide/execution/task_based_execution.html) for an example using SLURM.

## Gather results

Running the simulation units in this example would be too expensive for a
tutorial notebook. For the remainder of the tutorial, we therefore use a
completed `Warehouse` from the same small network.

We download the precomputed Warehouse from 
[Zenodo](https://doi.org/10.5281/zenodo.23072369).

In [14]:
if not Path("warehouse_mcl1_small.gz").is_file():
    !curl -fLO https://zenodo.org/records/23072369/files/warehouse_mcl1_small.gz
if not Path("warehouse_mcl1_small").is_dir():
    !tar -xzf warehouse_mcl1_small.gz

A `Warehouse` can be reopened using `FileSystemWarehouse.from_dir`:

In [15]:
from openfe.storage.warehouse import FileSystemWarehouse

example_wh = FileSystemWarehouse.from_dir("warehouse_mcl1_small")

The `Warehouse` now contains the `ProtocolUnitResult`s produced by completed tasks.

`gather_all_results` reconstructs DAG-level results and returns a
`(ProtocolResult, ProtocolDAGResult)` pair for each gathered `ProtocolDAG`.

For this RBFE network, these `ProtocolResult`s correspond to individual
**complex or solvent legs** of each edge. They are therefore leg-level
free-energy estimates, not yet the final relative binding free energies.

In [16]:
result_edges = example_wh.gather_all_results()

for protocol_result, dag_result in result_edges:
    name = example_wh.load_setup_tokenizable(
        dag_result.transformation_key
    ).name

    if dag_result.ok():
        print(
            f"{name}: "
            f"{protocol_result.get_estimate():.2f} ± "
            f"{protocol_result.get_uncertainty():.2f}"
        )
    else:
        print(f"{name}: failed")

INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges

ligand_2_complex_ligand_3_complex: 2.48 kilocalorie_per_mole ± 0.00 kilocalorie_per_mole
ligand_2_solvent_ligand_3_solvent: 1.59 kilocalorie_per_mole ± 0.00 kilocalorie_per_mole
ligand_1_complex_ligand_2_complex: 0.39 kilocalorie_per_mole ± 0.00 kilocalorie_per_mole
ligand_1_solvent_ligand_2_solvent: 3.48 kilocalorie_per_mole ± 0.00 kilocalorie_per_mole


### Analysing results with `openfe gather`

Until task-based execution has its own results command, we use `warehouse_to_legacy_json` to write results in the same format as `openfe quickrun`, one JSON file per `Transformation`. The command-line equivalent is `openfe to-legacy-json example_results/warehouse_mcl1_small/ -o results_mcl1`.

In [17]:
from openfe.storage.utils import warehouse_to_legacy_json

warehouse_to_legacy_json(result_edges=result_edges, out_dir="mcl1_result_jsons")

The output directory can then be passed to `openfe gather`, which combines the complex and solvent legs into relative binding free energies:

In [18]:
!openfe gather mcl1_result_jsons --report ddg

/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=14846) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Traceback (most recent call last):
  File "/Users/atravitz/micromamba/envs/openfe-notebooks/bin/openfe", line 3, in <module>
    from openfecli.cli import main
  File "/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openfecli/cli.py", line 35, in <module>
    @click.command(cls=OpenFECLI, name="openfe", help=_MAIN_HELP, context_settings=CONTEXT_SETTINGS)
     ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/click/decorators.py", line 248, in decorator
    cmd = cls(name=cmd_name, callback=f, params=params, **attrs)
  File "/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/plugcli/cli.py", line 30, in __init__
    self._register_plugin(plugin)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/plugcli/cli.py", line 51, in _register

## Summary

In this tutorial, we:

- created a task-based campaign from an `AlchemicalNetwork` with `setup_task_campaign` which created the `Warehouse` and the `TaskStatusDB`
- used a `Worker` to claim and execute available tasks
- inspected how task states change during execution
- reopened a completed `Warehouse` and gathered `ProtocolResult`s with `gather_all_results()`.

The result is a workflow in which **openfe** manages task dependencies and shared campaign data while workers focus on executing available tasks.